# LoRa Path Loss ML Analysis

Machine learning classification of path loss (high vs. low) from LoRa RSSI measurements. Uses 5 anchor receivers and classical ML models.

In [ ]:
# Uncomment for Google Colab:
# !pip install numpy pandas matplotlib seaborn scikit-learn --quiet

# Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_curve, auc, confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

# Plot style
sns.set_style("whitegrid")
plt.rcParams.update({
    'figure.dpi': 130,
    'font.family': 'serif',
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
})
print("All libraries imported successfully.")

# Configuration

In [ ]:
# Data directory: tries local paths first, then Colab /content/
def get_data_dir():
    candidates = [
        os.path.join(os.getcwd(), 'data', 'raw'),
        os.path.join(os.getcwd(), '..', 'data', 'raw'),
        os.path.join(os.getcwd(), '..', 'Path Loss Measurements'),
        '/content/',
    ]
    for d in candidates:
        p = os.path.join(d, 'Anchor 1.txt')
        if os.path.exists(p):
            return d
    return os.path.join(os.getcwd(), 'data', 'raw')

# Output for figures (project root or notebooks/)
cwd = os.getcwd()
if cwd.endswith('notebooks'):
    OUTPUT_DIR = os.path.join(cwd, 'output')
else:
    OUTPUT_DIR = os.path.join(cwd, 'output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

DATA_DIR = get_data_dir()
print(f"📂 Data directory: {DATA_DIR}")
print(f"📁 Output directory: {OUTPUT_DIR}")

# Load Dataset

In [ ]:
# Load Dataset
print("Loading and combining data from anchor files...")

all_data = []
for i in range(1, 6):
    file_path = os.path.join(DATA_DIR, f'Anchor {i}.txt')
    try:
        with open(file_path, 'r') as f:
            # Extract rssi values using regex
            rssi_values = []
            for line in f:
                line = line.strip()
                if not line or line.startswith('ANCHOR'):
                    continue
                match = re.search(r'rssi=(-?\d+)', line)
                if match:
                    try:
                        rssi_values.append(float(match.group(1)))
                    except ValueError:
                        pass


        for j, rssi in enumerate(rssi_values):
            all_data.append({
                'receiver_id': i,
                'measurement_point_id': j, # This is a proxy for spatial location
                'path_loss': -rssi # Use negative RSSI as a proxy for path loss
            })
    except FileNotFoundError:
        print(f"Error: File not found at {file_path}. Please ensure all processed data files are uploaded correctly.")
        raise FileNotFoundError(f"Missing file: {file_path}")

df = pd.DataFrame(all_data)
print(f"Combined DataFrame shape: {df.shape}")
print("First 5 rows of the combined DataFrame:")
print(df.head())

# --- 2. Feature Engineering --- #
# For classification, we need a target variable. Let\'s create a hypothetical one.
# For instance, classify if path loss is \'high\' or \'low\' based on a median split.
# In a real scenario, the target variable would come from the problem definition (e.g., localization error, connectivity status).

median_pl = df['path_loss'].median()
df['path_loss_category'] = (df['path_loss'] > median_pl).astype(int) # 1 for high, 0 for low

print(f"Median Path Loss: {median_pl:.2f}")
print("Distribution of Path Loss Categories:")
print(df['path_loss_category'].value_counts())

# Let\'s also create some interaction features or polynomial features if relevant
df['receiver_pl_interaction'] = df['receiver_id'] * df['path_loss']
df['measurement_point_pl_interaction'] = df['measurement_point_id'] * df['path_loss']

# Define features (X) and target (y)
X = df[['receiver_id', 'measurement_point_id', 'path_loss', 'receiver_pl_interaction', 'measurement_point_pl_interaction']]
y = df['path_loss_category']

print("Features (X) and Target (y) prepared.")
print("X head:")
print(X.head())
print("y head:")
print(y.head())

# --- 3. Data Scaling --- #
# It\'s good practice to scale features for many ML models
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

print("Features scaled.")
print("X_scaled_df head:")
print(X_scaled_df.head())

# --- 4. Machine Learning Models and Evaluation --- #

# Define models
models = {
    "Logistic Regression": LogisticRegression(random_state=42, solver='liblinear'),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "Support Vector Machine": SVC(probability=True, random_state=42) # probability=True for ROC curve
}

results = []

# Define train-test splits
splits = {"50/50 Split": 0.5, "60/40 Split": 0.4} # test_size

for split_name, test_size in splits.items():
    print(f"\n--- {split_name} ---")
    X_train, X_test, y_train, y_test = train_test_split(X_scaled_df, y, test_size=test_size, random_state=42, stratify=y)

    for model_name, model in models.items():
        print(f"Training {model_name}...")
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        print(f"  Accuracy: {accuracy:.4f}")
        print(f"  Precision: {precision:.4f}")
        print(f"  Recall: {recall:.4f}")
        print(f"  F1-Score: {f1:.4f}")

        results.append({
            "Split": split_name,
            "Model": model_name,
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "F1-Score": f1
        })

        # --- Feature Importance (for tree-based models) ---
        if hasattr(model, "feature_importances_"):
            feature_importance = pd.Series(model.feature_importances_, index=X_scaled_df.columns).sort_values(ascending=False)
            plt.figure(figsize=(10, 6))
            sns.barplot(x=feature_importance, y=feature_importance.index, palette="viridis")
            plt.title(f"Feature Importance for {model_name} ({split_name})")
            plt.xlabel("Importance")
            plt.ylabel("Feature")
            plt.tight_layout()
            plt.savefig(os.path.join(OUTPUT_DIR, f"feature_importance_{model_name.replace(' ', '_')}_{split_name.replace('/', '_')}.png"), dpi=300)
            plt.close()
            print(f"  Saved feature importance plot for {model_name} ({split_name})")

        # --- ROC Curve ---
        if hasattr(model, "predict_proba"):
            y_proba = model.predict_proba(X_test)[:, 1]
            fpr, tpr, _ = roc_curve(y_test, y_proba)
            roc_auc = auc(fpr, tpr)

            plt.figure(figsize=(8, 8))
            plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
            plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
            plt.xlim([0.0, 1.0])
            plt.ylim([0.0, 1.05])
            plt.xlabel('False Positive Rate')
            plt.ylabel('True Positive Rate')
            plt.title(f'Receiver Operating Characteristic for {model_name} ({split_name})')
            plt.legend(loc='lower right')
            plt.tight_layout()
            plt.savefig(os.path.join(OUTPUT_DIR, f"roc_curve_{model_name.replace(' ', '_')}_{split_name.replace('/', '_')}.png"), dpi=300)
            plt.close()
            print(f"  Saved ROC curve for {model_name} ({split_name})")

        # --- Confusion Matrix ---
        cm = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                    xticklabels=['Low PL', 'High PL'], yticklabels=['Low PL', 'High PL'])
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.title(f'Confusion Matrix for {model_name} ({split_name})')
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, f"confusion_matrix_{model_name.replace(' ', '_')}_{split_name.replace('/', '_')}.png"), dpi=300)
        plt.close()
        print(f"  Saved confusion matrix for {model_name} ({split_name})")

# --- 5. Correlation Map --- #
print("\nGenerating correlation map...")
plt.figure(figsize=(12, 10))
sns.heatmap(X_scaled_df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Feature Correlation Map")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "feature_correlation_map.png"), dpi=300)
plt.close()
print("Saved feature correlation map.")

# --- 6. Summarize Results --- #
results_df = pd.DataFrame(results)
print("\n--- Summary of Model Performance ---")
print(results_df)

results_df.to_csv(os.path.join(OUTPUT_DIR, "model_performance_summary.csv"), index=False)
print("Model performance summary saved to model_performance_summary.csv")

print("Machine learning analysis complete. All plots and summary saved.")

Loading and combining data from anchor files...
Error: File not found at /content/Anchor 1.txt. Please ensure all processed data files are uploaded correctly.


FileNotFoundError: Missing file: /content/Anchor 1.txt